[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/Modulo%204%3A%20NLP/04_OCR_y_tagging.ipynb)


# 04 — PDF IMSS: OCR, comparación y **tagging**

1. Bajar el PDF y generar **`output/texto_pypdf.txt`**, **`texto_easyocr.txt`** y (opcional) tablas.
2. **Tagging en dos bloques separados**:
   - **A — Montos:** regex solo para cantidades con **`$`**, más el texto **entre paréntesis** (cantidad en letras). Resultado **por página** (tabla + resumen + JSON).
   - **B — Personas:** **NER** (`spaCy`), solo etiqueta **`PER`** (nombres propios; el modelo no separa género en la etiqueta).
3. **(Opcional, más abajo)** Frases **retadoras**, NER sobre el PDF con **`PAGINA_FINAL`** (solo un tramo del documento), y **entrenar un NER propio** (`CANTIDAD` / `PRODUCTO`) con pruebas en varias oraciones.


## 1) Texto con **pypdf** → `output/texto_pypdf.txt`

In [ ]:
from pypdf import PdfReader
from pathlib import Path
from subprocess import run

ruta_pdf = "data/006400001N00225-001-00_Censurado.pdf"
CARPETA_SALIDA = Path("output")
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

lector = PdfReader(ruta_pdf)
partes = []
for i, pagina in enumerate(lector.pages):
    partes.append(f"--- página {i + 1} ---\n")
    partes.append(pagina.extract_text() or "")
texto_pypdf = "\n".join(partes).strip()

salida_pypdf = CARPETA_SALIDA / "texto_pypdf.txt"
salida_pypdf.write_text(texto_pypdf, encoding="utf-8")
print(salida_pypdf.resolve(), "—", len(texto_pypdf), "caracteres")

## 2) Texto con **EasyOCR** (páginas como imagen) → `output/texto_easyocr.txt`

In [ ]:
# import easyocr
# import fitz
# import numpy as np
# from PIL import Image

# IDIOMAS = ["es", "en"]
# lector_ocr = easyocr.Reader(IDIOMAS, gpu=False)

# doc = fitz.open(ruta_pdf)
# n = min(doc.page_count, 10)  # solo primeras 10 páginas (OCR lento)
# bloques = []
# for i in range(n):
#     pix = doc.load_page(i).get_pixmap(matrix=fitz.Matrix(2, 2))
#     img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
#     det = lector_ocr.readtext(np.asarray(img), detail=1)
#     bloques.append(f"--- página {i + 1} ---\n" + "\n".join(t[1] for t in det))
# doc.close()

# texto_ocr = "\n\n".join(bloques).strip()
# salida_ocr = CARPETA_SALIDA / "texto_easyocr.txt"
# salida_ocr.write_text(texto_ocr, encoding="utf-8")
# print(salida_ocr.resolve(), "—", len(texto_ocr), "caracteres", f"({n} páginas OCR)")

## 3) (Opcional) **Tablas** con **pdfplumber** → `output/texto_tablas_pdfplumber.txt`

Intenta detectar tablas página por página y las vuelca como texto separado por `|` y líneas.

In [ ]:
from pathlib import Path

CARPETA_SALIDA = Path("output")
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

import pdfplumber

lineas = []
with pdfplumber.open(ruta_pdf) as pdf:
    for i, pagina in enumerate(pdf.pages):
        lineas.append(f"--- página {i + 1} ---")
        tablas = pagina.extract_tables() or []
        if not tablas:
            lineas.append("(sin tablas detectadas en esta página)")
            continue
        for t_idx, tabla in enumerate(tablas):
            lineas.append(f"-- tabla {t_idx + 1} --")
            for fila in tabla:
                celdas = [str(c) if c is not None else "" for c in fila]
                lineas.append(" | ".join(celdas))

texto_tablas = "\n".join(lineas)
salida_tablas = CARPETA_SALIDA / "texto_tablas_pdfplumber.txt"
salida_tablas.write_text(texto_tablas, encoding="utf-8")
print(salida_tablas.resolve(), "—", len(texto_tablas), "caracteres")

## Resumen

Abre en el editor **`output/texto_pypdf.txt`** y **`output/texto_easyocr.txt`** y compara. Si `pypdf` está casi vacío y EasyOCR lleno, el PDF es sobre todo imagen escaneada.

---

## Cómo encajan **regex** y **NER**

| Qué quieres | Herramienta | Por qué |
|-------------|-------------|---------|
| Cantidades en pesos / dólares, formatos `1,234.56`, `$…`, `MN`… | **Regex** (`re`) | Los formatos se describen con reglas; no hace falta un modelo grande. |
| Quién firmó, qué empresa, instituciones, a veces fechas | **NER** | El modelo aprendió qué trozos de texto “parecen” persona u organización en español. |

**No compiten:** en un pipeline típico usas **ambos** sobre el mismo texto (el de `pypdf` o el de EasyOCR, el que mejor se lea).

**NER en la práctica (spaCy):** etiquetas útiles suelen ser `PER` (persona), `ORG` (organización), `LOC` (lugar); según el modelo puede haber más. No es magia: a veces confunde cargos con nombres; por eso en clase conviene **revisar** la lista.

**Instalación del modelo en español** (una vez en terminal o con `!python -m spacy download es_core_news_sm`):

```text
python -m spacy download es_core_news_sm
```

El sufijo `sm` = modelo pequeño (rápido). Para más calidad existe `md` / `lg` (más pesados).

## Tagging A — **Montos** (regex con **signo de pesos**)

Buscamos líneas del tipo `$8,800,000.00 (OCHO MILLONES … M.N.)`: el **número con `$`** y, si viene pegado, el **texto entre paréntesis**.

**Enfoque pedagógico:** primero probamos la **misma regla** en **una frase de ejemplo** (celdas siguientes). Cuando veas cómo encaja el patrón, la celda de abajo aplica **exactamente esa idea** a **`output/texto_pypdf.txt`** y guarda **`output/montos_por_pagina.json`**.


### Mini-ejemplo: **regex** en una frase

Misma regla que la celda de abajo: el **texto** y los **montos** que encuentra el patrón.


In [ ]:
import re

PAT_MONTO = re.compile(
    r"\$\s*[\d]{1,3}(?:,\d{3})*(?:\.\d{2})\s*(\([^)]+\))?",
    re.MULTILINE,
)

texto = "Pagar $1,250.50 hoy y luego $8,800,000.00 (OCHO MILLONES...) mañana."
texto, [m.group(0) for m in PAT_MONTO.finditer(texto)]


In [ ]:
import json
import re
from pathlib import Path

import pandas as pd

from IPython.display import display

CARPETA_SALIDA = Path("output")
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
RUTA_TEXTO = CARPETA_SALIDA / "texto_pypdf.txt"
texto = RUTA_TEXTO.read_text(encoding="utf-8")

# -----------------------------------------------------------------------------
# 1. Regla (regex): qué buscamos
#    - Empieza con $
#    - Número con comas de miles y dos decimales (ej. 8,800,000.00)
#    - Opcional: espacio y (… cantidad en letras …) como en el contrato IMSS
#    Si tu PDF usa otro formato, solo tocas PAT_MONTO aquí.
# -----------------------------------------------------------------------------
PAT_MONTO = re.compile(
    r"\$\s*[\d]{1,3}(?:,\d{3})*(?:\.\d{2})\s*(\([^)]+\))?",
    re.MULTILINE,
)

# -----------------------------------------------------------------------------
# 2. Cómo se parte el documento en "páginas"
#    pypdf ya metió marcas --- página N ---; esta función devuelve (n, texto).
# -----------------------------------------------------------------------------
def iter_paginas(contenido: str):
    marcas = list(re.finditer(r"---\s*página\s+(\d+)\s*---", contenido, flags=re.IGNORECASE))
    for i, m in enumerate(marcas):
        n = int(m.group(1))
        inicio = m.end()
        fin = marcas[i + 1].start() if i + 1 < len(marcas) else len(contenido)
        yield n, contenido[inicio:fin].strip()

# -----------------------------------------------------------------------------
# 3. Proceso: recorrer página por página y aplicar la regla
# -----------------------------------------------------------------------------
filas = []
por_pagina = []

for num_pag, bloque in iter_paginas(texto):
    hallazgos = []
    for m in PAT_MONTO.finditer(bloque):
        trozo = m.group(0).strip()
        en_palabras = (m.group(1) or "").strip()
        num_m = re.search(r"\$\s*[\d,]+\.\d{2}", trozo)
        monto_dolares = num_m.group(0).replace(" ", "") if num_m else trozo.split("(")[0].strip()
        filas.append(
            {"página": num_pag, "monto_$": monto_dolares, "entre_paréntesis": en_palabras}
        )
        hallazgos.append({"monto_$": monto_dolares, "entre_paréntesis": en_palabras})
    por_pagina.append({"página": num_pag, "montos": hallazgos})

# -----------------------------------------------------------------------------
# 4. Salida: archivo JSON en output/ + tabla + resumen en pantalla
# -----------------------------------------------------------------------------
salida_m = {"archivo_fuente": str(RUTA_TEXTO), "por_página": por_pagina}
(CARPETA_SALIDA / "montos_por_pagina.json").write_text(
    json.dumps(salida_m, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

df = pd.DataFrame(filas)
display(df)

print("\n--- Resumen por página ---\n")
for item in por_pagina:
    p = item["página"]
    lista = item["montos"]
    if not lista:
        continue
    print(f"Página {p}: se mencionan {len(lista)} monto(s) con $")
    for h in lista:
        pal = h["entre_paréntesis"]
        recorte = pal[:100] + ("…" if len(pal) > 100 else "")
        print("  ·", h["monto_$"], recorte)
    print()


## Tagging B — **Personas** (NER con spaCy)

Solo etiqueta **`PER`** (*person*): nombres propios que el modelo reconoce (hombre o mujer; el modelo no distingue por género en la etiqueta). Resultado **por página**.

**Antes** de procesar el PDF, las celdas siguientes muestran **qué es el NER** con una oración corta. **Después**, la última celda aplica el mismo modelo a **`output/texto_pypdf.txt`**.

La primera vez necesitas: `python -m spacy download es_core_news_sm`


### Mini-ejemplo: **NER** en una oración

Mismo modelo que la celda de abajo: lista de **textos**, **filtrado** por etiquetas (`if ent.label_ in categorias`) como en los tutoriales de spaCy, y luego **displacy** + dependencias en la primera oración.


In [ ]:
# %pip install spacy blis thinc --force-reinstall --no-cache-dir

### ¿Qué es `es_core_news_sm`?

El nombre del modelo no es aleatorio — cada parte indica exactamente qué contiene:

| Parte | Significado |
|-------|-------------|
| `es` | **Español** — idioma del modelo |
| `core` | Modelo **completo**: incluye tokenización, análisis gramatical (POS), dependencias sintácticas y NER |
| `news` | Entrenado con **textos de noticias** en español |
| `sm` | **Small** — versión pequeña y rápida |

---

#### ¿Qué tamaños existen?

spaCy ofrece tres tamaños para el modelo en español:

```
es_core_news_sm   # ~12 MB  → rápido, ideal para clase y prototipos
es_core_news_md   # ~43 MB  → incluye vectores de palabras (similitud semántica)
es_core_news_lg   # ~567 MB → más preciso en NER y análisis
```

> **Regla práctica:** empieza con `sm`. Si la precisión no es suficiente para tu caso de uso, sube a `md` o `lg`.

---

#### ¿Cómo instalo otro modelo?

```bash
python -m spacy download es_core_news_lg
```

Y en código solo cambias el nombre:

```python
nlp = spacy.load("es_core_news_lg")
```

Catálogo completo de modelos (todos los idiomas): https://spacy.io/models

### spaCy no es “una sola cosa”: pipeline, NER, POS, dependencias y morfología

**Idea clave para el salón:** `nlp(texto)` devuelve un `Doc` con **varias capas de análisis**. El **NER** (entidades con `PER`, `ORG`, `LOC`…) es **solo una** de esas capas. Las otras sirven para gramática y sintaxis; conviven en el mismo modelo `es_core_news_*` pero **no son el mismo algoritmo** que el NER.

| Capa | Qué hace | Dónde lo ves en spaCy |
|------|----------|------------------------|
| **Tokenización** | Parte el texto en tokens (palabras, signos…) | `for token in doc` |
| **Etiquetado POS (UPOS)** | Asigna categoría gramatical *gruesa* (sustantivo, verbo, determinante…) | `token.pos_`, `token.tag_` |
| **Morfología** | Rasgos como género, número, tiempo verbal (cuando el modelo los predice) | `token.morph` |
| **Dependencias sintácticas** | Cada token tiene un **head** y una **relación** (`dep_`); forma un **árbol** | `token.head`, `token.dep_` |
| **NER** | Detecta **fragmentos** (varios tokens seguidos) como entidad con **tipo** | `doc.ents`, `ent.label_` |

#### ¿Qué es **UD** (Universal Dependencies)?

Es un **estándar** (guías y etiquetas) para anotar **dependencias** y alinear **POS** entre idiomas. Los modelos de español de spaCy suelen estar entrenados con **treebanks** compatibles con UD (o muy cercanos). Eso **no** significa que “UD piense” la oración: significa que las etiquetas que ves (`nsubj`, `obj`, `ROOT`…) son **nombres convencionales** acordados internacionalmente.

#### Cómo se relacionan las cuatro ideas con la oración de ejemplo

Usamos la misma oración que en la celda de abajo. **Sin inventar** el análisis token a token. A nivel *conceptual*:

1. **POS** — Palabras como *La*, *doctora*, *firmaron*, *convenio* reciben etiquetas de categoría (determinante, sustantivo, verbo, etc.). Ayuda al modelo (y a nosotros) a saber *qué papel gramatical* cumple cada pieza.

2. **Morfología** — En verbos como *firmaron* puede aparecer plural/persona/tiempo; en sustantivos/adjetivos a veces género y número. **Depende** de qué prediga el modelo para cada token.

3. **Dependencias** — Se construye un **árbol dirigido**: casi todos los tokens **cuelgan** de otro token (su `head`), hasta llegar a un único **`ROOT`** por oración (en oraciones compuestas hay matices, pero la idea pedagógica es: *hay un nodo raíz del análisis principal*).

4. **NER** — Busca **nombres propios y similares** y los agrupa con etiquetas (`PER`, `ORG`, `LOC`…). Es **independiente** del árbol de dependencias: puedes tener un `PER` perfecto y a la vez un error en `obj` si el parser se equivoca.

#### ¿**`firmaron` → `ROOT`** es cierto? ¿Cómo “lo sabe”?

En **español**, en una oración **declarativa** como esta, el **verbo finito principal** del predicado suele ser el **`ROOT`**: los sintacticianos lo anotan así en los corpus de entrenamiento y el **parser estadístico** aprende a reproducir ese patrón. **No** es que spaCy aplique una regla manual del tipo “el verbo siempre es raíz”; es que el modelo fue entrenado con **miles de oraciones** ya anotadas y **predice** el árbol más probable.

- **Puede equivocarse** en oraciones largas, raras o mal tokenizadas.
- Para **demostrarlo sin discutir**, en la celda siguiente, además del NER, se muestra el árbol de dependencias; revisa el token *firmaron* y su `dep_`.

#### Comprobar en código (sin fe ciega)

Tras `doc = nlp(oracion)`:

```python
for t in doc:
    if t.dep_ == "ROOT" or t.text == "firmaron":
        print(t.text, "→", t.dep_, "| head:", t.head.text)
```

Y la visualización `displacy.render(doc, style="dep", jupyter=True)` (incluida en la celda de demo) muestra el árbol completo.

**Sobre la línea impresa `head='firmaron'`:** en el token con `dep_ == "ROOT"`, spaCy suele hacer que el token sea **su propio head** (`token.head == token`). No es un error: significa “este es el nodo raíz del árbol”; el resto de palabras cuelgan (directa o indirectamente) de ahí.


In [ ]:
import spacy
from spacy import displacy

nlp_demo = spacy.load("es_core_news_sm")

# Patrón tipo tutorial (lista de textos → filtrar NER por etiquetas)
# En modelos en inglés suele ser PERSON; en es_core_news_* la persona es PER.
# Dos oraciones solo para ver el bucle con más de un ejemplo; la segunda no tiene que ver con el PDF del IMSS.
textos = [
    (
        "La doctora Ana Pérez y el ingeniero Carlos Ruiz firmaron un convenio entre el IMSS "
        "y Microsoft México el pasado lunes en las oficinas de Guadalajara, "
        "representando al gobierno mexicano ante la ONU."
    ),
    "Convocaron a los participantes en la Ibero Ciudad de México y al día siguiente sesionaron en Guadalajara.",
]
categorias = ["ORG", "PER", "LOC"]

# Todas las etiquetas NER que expone este modelo (opcional):
# print(tuple(nlp_demo.get_pipe("ner").labels))

docs = [nlp_demo(t) for t in textos]
for i, doc_i in enumerate(docs):
    entidades = [(e.text.strip(), e.label_) for e in doc_i.ents if e.label_ in categorias]
    print(f"--- Oración {i + 1} — entidades filtradas ---")
    print(entidades)
    print()

# Visualización en la primera oración (la del ejemplo largo)
doc = docs[0]

# NER: entidades (PER, ORG, LOC, …)
displacy.render(doc, style="ent", jupyter=True)

# Dependencias: comprobar quién es ROOT (suele ser el verbo principal de la oración matriz)
displacy.render(doc, style="dep", jupyter=True)

# Mismos gráficos a disco (SVG en `output/`; ábrelos en el navegador o en el editor)
from pathlib import Path as _Path

_carp = _Path("output")
_carp.mkdir(parents=True, exist_ok=True)
for _estilo, _arch in (("ent", "displacy_demo_ent.svg"), ("dep", "displacy_demo_dep.svg")):
    _svg = displacy.render(doc, style=_estilo, jupyter=False)
    (_carp / _arch).write_text(_svg, encoding="utf-8")
print("SVG guardados:", str(_carp.resolve() / "displacy_demo_ent.svg"), "y", str(_carp.resolve() / "displacy_demo_dep.svg"))

for t in doc:
    if t.dep_ == "ROOT" or t.text.lower() == "firmaron":
        print(f"{t.text!r} → dep={t.dep_!r} | head={t.head.text!r}")

#### Etiquetas del gráfico de **dependencias** (resumen breve)

Son nombres del estándar **Universal Dependencies** (UD). Cada arco une un **head** con un **dependent**; la etiqueta dice *qué papel* cumple el **dependent** respecto al **head**.

| Etiqueta | Significado (breve) |
|----------|---------------------|
| **acl** | Cláusula adjetiva que modifica un sustantivo (*el hecho [de que…]*). |
| **acl:relcl** | Cláusula de relativo (*la persona **que** vino*). |
| **advcl** | Cláusula adverbial (*… **cuando** llegó*). |
| **advmod** | Adverbio que modifica verbo/adjetivo/adverbio (*muy* rápido). |
| **amod** | Adjetivo que modifica un sustantivo (*gobierno **mexicano***). |
| **appos** | Apósito: nombre al lado de otro que lo explica (*doctora **Ana***). |
| **aux** | Auxiliar del verbo principal (*ha* venido). |
| **case** | Marca de caso: preposición o partícula que liga un sintagma (*en* la mesa, *al* gobierno). |
| **cc** | Conjunción coordinante (*y*, *o*, *pero*). |
| **ccomp** | Cláusula complemento con sujeto propio (*dijo **que** sí*). |
| **compound** | Forma compuesta (*tren* + *balas*). |
| **conj** | Segundo miembro de una coordinación (hermano de otro *conj* o del primer término). |
| **cop** | Copulativa (*es* alto). |
| **csubj** | Cláusula sujeto (***Que** llueva* molesta). |
| **dep** | Relación no clasificada con más detalle (residuo). |
| **det** | Determinante (*el*, *un*, *la*). |
| **discourse** | Elemento discursivo (interjección, etc.). |
| **dislocated** | Tema desplazado al inicio/final (*Eso*, no lo creo). |
| **expl** | Pronominal explético (*se* en *se dice*). |
| **fixed** | Grupo fijo multi-palabra (*en* *cuanto* a). |
| **flat** | Nombres u otros elementos sin jerarquía interna clara (*Ana* **Pérez**, *Microsoft* **México**). |
| **goeswith** | Partes mal tokenizadas que van juntas. |
| **iobj** | Objeto indirecto (p.ej. *le* di un libro). |
| **list** | Elementos de lista. |
| **mark** | Marcador subordinante (*que*, *si*, *porque*). |
| **nmod** | Modificador nominal (sustantivo/adjetivo/prep. que añade información a un sustantivo). |
| **nsubj** | Sujeto nominal (*ellos* firmaron). |
| **nsubj:pass** | Sujeto de pasiva (*fue firmado* **por** ellos). |
| **nummod** | Número que modifica (*tres* libros). |
| **obj** | Objeto directo (complemento principal del verbo transitivo). |
| **obl** | Oblicuo: complemento preposicional o similar (*ante* la ONU). |
| **obl:unmarked** | Oblicuo sin preposición explícita (según idioma/corpus). |
| **orphan** | Hijo “huérfano” en elipsis coordinada. |
| **parataxis** | Dos cláusulas yuxtapuestas sin conector claro. |
| **punct** | Signo de puntuación. |
| **ref** | Pronombre co-referente con el antecedente. |
| **root** | Raíz: el nodo principal del árbol (casi siempre el verbo de la oración). |
| **vocative** | Vocativo (*Doctor*, espere). |
| **xcomp** | Complemento sin sujeto propio obligatorio (*quiere* **ir**). |

> No todas aparecen en cada oración; el modelo solo muestra las que usa para **esa** frase. Si una etiqueta no sale en tu gráfica, sigue siendo útil conocer el catálogo.


### Documento IMSS: NER sobre el PDF (muestra)

Aquí aplicamos el **mismo** `es_core_news_sm` a `output/texto_pypdf.txt`, **solo en las primeras páginas** (cambia `PAGINA_FINAL` si quieres ver más o menos).

Esta parte va **antes** del entrenamiento de un NER propio más abajo: primero observamos el modelo general; después el mini-modelo `CANTIDAD`/`PRODUCTO`.


In [ ]:
import json, re
from pathlib import Path
import pandas as pd
import spacy
from spacy import displacy
from IPython.display import display, HTML

# Solo las primeras PAGINA_FINAL páginas (según la marca --- página N --- de pypdf)
PAGINA_FINAL = 20

CARPETA_SALIDA = Path("output")
texto = (CARPETA_SALIDA / "texto_pypdf.txt").read_text(encoding="utf-8")
nlp = spacy.load("es_core_news_sm")


def iter_paginas(contenido):
    marcas = list(re.finditer(r"---\s*página\s+(\d+)\s*---", contenido, flags=re.IGNORECASE))
    for i, m in enumerate(marcas):
        n = int(m.group(1))
        inicio = m.end()
        fin = marcas[i + 1].start() if i + 1 < len(marcas) else len(contenido)
        yield n, contenido[inicio:fin].strip()


filas = []
por_pagina = []

for num_pag, bloque in iter_paginas(texto):
    if num_pag > PAGINA_FINAL:
        break
    if not bloque.strip():
        continue
    doc = nlp(bloque[:50_000])  # displacy necesita docs más cortos
    entidades = [(ent.text.strip(), ent.label_) for ent in doc.ents if ent.text.strip()]
    vistos = set()
    nombres_unicos = []
    for texto_ent, label in entidades:
        if texto_ent not in vistos:
            vistos.add(texto_ent)
            filas.append({"página": num_pag, "entidad": texto_ent, "tipo": label})
            if label == "PER":
                nombres_unicos.append(texto_ent)
    por_pagina.append({"página": num_pag, "doc": doc, "personas": nombres_unicos})

# ── Tabla resumen ─────────────────────────────────────────────────────────────
display(HTML(f"<h3>Tabla de entidades por página (páginas 1–{PAGINA_FINAL})</h3>"))
display(pd.DataFrame(filas))

# ── Visualización con colores por página ─────────────────────────────────────
display(HTML("<h3>Entidades resaltadas por página</h3>"))
for item in por_pagina:
    p = item["página"]
    doc = item["doc"]
    if not doc.ents:
        continue
    display(HTML(f"<h4 style='margin-top:1.5em;color:#444'>📄 Página {p}</h4>"))
    displacy.render(doc, style="ent", jupyter=True)


---

## Extra (opcional): **entrenar un NER propio** con spaCy

1. Los datos de entrenamieto son una **lista de tuplas**: `(texto, {"entities": [(inicio, fin, "ETIQUETA"), ...]})`.
2. Los números `inicio` y `fin` son **índices de caracteres** en el string Python (como si fuera un arreglo: empieza en 0).
3. Se entrena con objetos **`Example`** y un bucle por **épocas** y **lotes** (`minibatch`).

Usaremos `spacy.blank("es")` para **no** partir de `es_core_news_*`: así el NER se entrena **desde cero** y no se mezclan pesos ya entrenados con otras tareas.


### 1) Formato de los datos (qué va en cada tupla)

- **`texto`**: la oración **exacta** tal como la verá el modelo.
- **`"entities"`**: lista de **ternas** `(inicio, fin, etiqueta)`.
  - `inicio` incluido, `fin` **excluido** (como en slices de Python: `texto[inicio:fin]`).
- Si te equivocas en un solo índice, el entrenamiento falla o aprende basura: por eso en la siguiente celda usamos un **ayudante** que calcula posiciones a partir del **fragmento literal**.


In [ ]:
def ejemplo_entrenamiento(oracion: str, trozos: list[tuple[str, str]]):
    """Devuelve la tupla que espera spaCy. trozos = [(texto_exacto, ETIQUETA), ...]"""
    entidades = []
    for texto, etiqueta in trozos:
        i = oracion.index(texto)
        entidades.append((i, i + len(texto), etiqueta))
    return (oracion, {"entities": entidades})


# Datos de ejemplo en español (pocos: en la práctica harían falta muchos más)
TRAIN_DATA = [
    ejemplo_entrenamiento(
        "¿Cuánto cuestan 10 libros de texto?",
        [("10", "CANTIDAD"), ("libros de texto", "PRODUCTO")],
    ),
    ejemplo_entrenamiento(
        "Necesitamos 5 proyectores para el auditorio.",
        [("5", "CANTIDAD"), ("proyectores", "PRODUCTO")],
    ),
    ejemplo_entrenamiento(
        "Pedimos 12 sillas y 3 mesas para el salón.",
        [("12", "CANTIDAD"), ("sillas", "PRODUCTO"), ("3", "CANTIDAD"), ("mesas", "PRODUCTO")],
    ),
    ejemplo_entrenamiento(
        "¿Puedes cotizar 2 impresoras y 7 toners?",
        [("2", "CANTIDAD"), ("impresoras", "PRODUCTO"), ("7", "CANTIDAD"), ("toners", "PRODUCTO")],
    ),
    ejemplo_entrenamiento(
        "Para el curso en la Ibero compramos 30 cuadernos.",
        [("30", "CANTIDAD"), ("cuadernos", "PRODUCTO")],
    ),
]

TRAIN_DATA[0]


### 2) Crear un modelo **vacío** en español y registrar **etiquetas nuevas**

- `spacy.blank("es")` solo tiene tokenización; nosotros añadimos el tubo **`ner`**.
- Cada etiqueta que aparezca en los datos (`CANTIDAD`, `PRODUCTO`, …) debe **registrarse** en el NER con `add_label` **antes** de inicializar pesos.


In [ ]:
import random

import spacy
from spacy.training import Example
from spacy.util import minibatch

# blank("es"): crea un pipeline *vacío* para español (idioma = reglas de tokenización).
# No carga es_core_news_*: no hay POS, dependencias ni NER preentrenados; solo añadimos "ner" abajo.
nlp_ent = spacy.blank("es")
ner = nlp_ent.add_pipe("ner", last=True)

for _, annotations in TRAIN_DATA:
    for _ini, _fin, etiqueta in annotations["entities"]:
        if etiqueta not in ner.labels:
            ner.add_label(etiqueta)

tuple(ner.labels)


### 3) Inicializar pesos y preparar el **optimizador**

- En spaCy 3, antes de `update` hay que **inicializar** el modelo con un iterador de `Example` (aquí los mismos datos de entrenamiento bastan para una demo).
- `resume_training()` devuelve el optimizador (`sgd`) que pasaremos a `nlp.update(...)` en cada lote.

> En tutoriales antiguos verás `nlp.begin_training()`: en spaCy 3 el flujo típico es `initialize` + `resume_training`.


In [ ]:
def a_ejemplos(nlp_obj, pares):
    for texto, ann in pares:
        doc = nlp_obj.make_doc(texto)
        yield Example.from_dict(doc, ann)


nlp_ent.initialize(lambda: a_ejemplos(nlp_ent, TRAIN_DATA))
optimizer = nlp_ent.resume_training()
optimizer


### 4) Bucle de entrenamiento: épocas, lotes y **pérdida** (`losses`)

- **`epochs`**: cuántas veces recorre todo el conjunto (en clase se usa un número bajo para no esperar mucho).
- **`minibatch`**: trocea los datos; cada trozo se convierte en `Example` y se llama a `nlp.update`.
- **`drop`**: “apagado” aleatorio de neuronas (regularización); evita memorizar de memoria las pocas frases.
- Imprimimos `losses` para ver si el error baja **en promedio** (con tan pocos datos puede rebotar).


In [ ]:
epochs = 40
random.seed(42)

for ep in range(epochs):
    random.shuffle(TRAIN_DATA)
    losses = {}
    lotes = minibatch(TRAIN_DATA, size=2)
    for lote in lotes:
        ejemplos = []
        for texto, annotations in lote:
            doc = nlp_ent.make_doc(texto)
            ejemplos.append(Example.from_dict(doc, annotations))
        nlp_ent.update(ejemplos, sgd=optimizer, losses=losses, drop=0.4)
    if (ep + 1) % 10 == 0 or ep == 0:
        print(f"época {ep + 1}: {losses}")


### 5) Guardar en disco, cargar y **probar** con texto nuevo

- `to_disk` crea una **carpeta** con el modelo (no un solo archivo `.bin` misterioso).
- Al probar, recuerda: este mini-modelo **solo** sabe de `CANTIDAD` y `PRODUCTO`; no esperes `PER` ni `ORG` como en `es_core_news_sm`.

En la siguiente celda hay un **ejercicio aparte**: varias oraciones de prueba para ver qué detecta el modelo ya entrenado.


In [ ]:
from pathlib import Path

from IPython.display import display, HTML
from spacy import displacy

CARPETA_MODELO = Path("output") / "mi_ner_cantidad_producto"
CARPETA_MODELO.mkdir(parents=True, exist_ok=True)
nlp_ent.to_disk(CARPETA_MODELO)

nlp_prueba = spacy.load(CARPETA_MODELO)

# ── Ejercicio aparte: varias oraciones de prueba (solo CANTIDAD / PRODUCTO) ──
# Misma idea que el NER del PDF (IMSS): resaltado con color vía displaCy.
frases_prueba = [
    "Para el diplomado pedimos 8 laptops y 15 ratones inalámbricos.",
    "¿Pueden traer 20 cuadernos y 4 estuches para el salón?",
    "Sin números ni productos claros: solo texto corrido.",
]

for i, frase in enumerate(frases_prueba, start=1):
    doc_p = nlp_prueba(frase)
    display(HTML(f"<h4 style='margin-top:1.25em;color:#444'>Prueba {i} — NER propio (CANTIDAD / PRODUCTO)</h4>"))
    displacy.render(doc_p, style="ent", jupyter=True)
    print("Lista bruta:", [(e.text, e.label_) for e in doc_p.ents])
